---

# **Introducción**

El **HydroSOS** es una iniciativa de la Organización Meteorológica Mundial (OMM) cuyo objetivo es aumentar la resistencia mundial a los riesgos hidroclimáticos y la gestión de los recursos hídricos mediante la elaboración de evaluaciones del estado y las perspectivas hidrológicas a diferentes escalas en todo el mundo. La idea es permitir la evaluación del estado actual de los recursos hídricos respecto a las condiciones "esperadas" o "normales", así como perspectivas hidrológicas de cómo este estado puede cambiar durante las siguientes semanas y meses. En el contexto de la iniciativa HydroSOS se ha creado un portal de demonstraciones de estado y perspectivas futuras, al que se puede acceder en: https://wmohydrosos.ceh.ac.uk/portal.

En este tutorial, aplicaremos la metodología HydroSOS para determinar el **estado hidrológico mensual** en un punto (e.g., estación hidrométrica) donde hay datos de caudales observados. La metodología consiste en transformar el caudal observado en un percentil relativo al histórico de la correspondiente época del año, por ejemplo, del mes en curso, y clasificar el percentil resultante según una categoría de flujo predeterminada.

Los scripts utilizados en este tutorial se basan en los códigos R (statuscalc.R) presentados en el repositorio oficial de HydroSOS (https://github.com/wmo-im/HydroSOS).

**Nota: Debido a actualizaciones metodológicas y al fin del soporte para el script R, se recomienda utilizar los scripts oficiales del repositorio oficial de HydroSOS en GitHub, como se presenta en la sección 8 de este cuaderno.**

---

# **Clonando el repositório del HydroSOS**

Clonaremos el repositório del WMO HydroSOS hospedado en Github para colectar una serie temporal ejemplo de caudales diarios.

*   Obs: Para utilizar el comando `git clone` en alguna otra plataforma de ejecución de notebooks/scripts python (e.g., ***Jupyter***, ***Visual Studio Code***), hay que certificarse de que el `Git` se encuentra instalado en su ordenador. En caso contrário, el git se puede descargar en (https://git-scm.com/downloads)

In [ ]:
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Importando librerías python**

Importaremos las librerías necesarias para la realización del tutorial:

In [ ]:
import numpy as np
import pandas as pd
import os

from datetime import datetime
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches
from pathlib import Path

---

# **Creando carpetas**

Crearemos automaticamente una carpeta output en el diretorio raiz donde vamos exportar los resultados:

In [ ]:
# Define folder path (e.g., in the current working directory)
folder_output = 'output'

# Create the folder if it doesn't exist
os.makedirs(folder_output, exist_ok=True)

---

# **1. Definición de los parámetros**

Aquí definiremos los parámetros para generar el estado hidrológico (*status*):

*   `File  path`: Camino donde se encuentra el archivo .csv con los caudales diarios
*   `min_valid_frac`: Fracción mínima de datos diários válidos en un mes para converterlos a promedio mensual
*   `clim_startYear`: Año inicial del período de referéncia para el cálculo de los percentiles
*   `clim_endYear`: Año final del período de referéncia para el cálculo de los percentiles
*   `perc_categories`: Lista de percentiles que indican los umbrales para las categorías hidrológicas  
*   `name_cateogories`: Lista de nombres para definición de las categorías
*   `color_codes` : Lista de colores para las categorías


In [ ]:
# Copy the path where input files are located
file_path = './HydroSOS/example_data/forecast/input/obs_dir/obs_1.csv' # Path to your CSV file

min_valid_frac = 0.5              # Minimum fraction of valid daily data per month (default = 50%)
clim_startYear = 1991             # Start year to compute statistics (mean, percentiles)
clim_endYear = 2020               # End year to compute statistics

# HydroSOS percentiles below of which (<=) flows are categorized (do not include 0.0)
perc_categories = [0.1, 0.25, 0.75, 0.9, 1.0]
name_categories = ['Muy abajo', 'Bajo', 'Normal', 'Alto', 'Muy alto']

# Color codes (HEX) according to HydroSOS standards
color_codes = ['#CD233f','#FFA885','#E7E2BC','#8ECEEE','#2C7DCD']

Los percentiles indicados podrían modificarse para adaptarse a las clasificaciones de interés del usuário, pero aquí mantendremos estos percentiles para facilitar la integración con los scripts existentes de HydroSOS.

---

# **2. Lectura de los datos**

2.1. Haremos la lectura del archivo .csv con los datos de caudales diários y los convertiremos a mensuales. El archivo .csv debe tener 2 columnas (una con las fechas, y otra con los valores de caudales), además de:

*   El titulo de la columna con las fechas definido como '*Date*' (sin aspas)
*   Las fechas en formato **YYYY-MM-DD**

In [ ]:
# Read and preprocess data
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df.set_index('Date', inplace=True)

# Ensure data comes in sorted position
df.sort_index(inplace=True)
df.index.name = 'Date'
# Ensure column name = Discharge
df.columns = ['Discharge']

# Convert to monthly data
df_month = df.resample('MS').mean()

2.2. Plotando el gráfico de los caudales:

In [ ]:
#@title Grafico
# Plot on screen (subplot for daily and monthly data)
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes = axes.flatten()

df.plot(ax=axes[0], color='k', linewidth=0.5, title='Diario')
df_month.plot(ax=axes[1], color='k', linewidth=0.5, title='Mensual')

for ax in axes:
  ax.set_ylabel('Caudal (m³/s)')
  ax.legend(['Observado'])

plt.tight_layout()
plt.show()

---

# **3. Pré-procesamiento de los datos mensuales**

3.1. Aquí aplicaremos algunos tratamientos básicos a los datos observados. Los datos no deben contener saltos en las fechas (por ejemplo, saltar de 1991-01-07 a 1991-01-09 sin el registro del día 08). Debemos asegurarnos de que la serie es continua y añadir las fechas que faltan (con datos inválidos) si es necesario:

In [ ]:
# Check if there are date differences between the full range of datetimes and datetimes informed
full_dates = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
diff = full_dates.difference(df.index)

# If there is a difference between the full range of datetimes and the range of dates informed, add the missing dates
if len(diff) > 0:
   for md in diff:
       # Add the missing dates and set nan values
       df.loc[md,'Discharge'] = np.nan

   # Sort values by date indices
   df.sort_index(inplace=True)

3.2. Removemos los meses con insuficiencia de datos diarios de acuerdo con el mínimo definido en la sección de parámetros:

In [ ]:
# Resample daily to monthly step and compute de fraction of dates in each month without missing data
valid_fraction = df.resample('MS').apply(lambda x: x.count() / x.size)

# Replace monthly values to NaN if % of valid < minimum
df_month.mask(valid_fraction < min_valid_frac, np.nan, inplace=True)

3.3. Podemos comprobar algunas estadísticas sobre los datos existentes (número total de años de la serie, % de datos no válidos):

In [ ]:
# Check whether or not there is enough data?
print(f"* Hay {df.index.year.max() - df.index.year.min() + 1} años en este archivo.")
print(f"* Hay {df.isna().sum(axis=0).values} datos inválidos (faltantes),")
print(f"  lo que corresponde a {100 * (df.isna().sum(axis=0)/len(df)).values}% del total de datos")

---

# **4. Cálculo del caudal como porcentaje del promedio**

4.1. Inicialmente, calculamos el caudal promedio de cada mes del calendario a lo largo de nuestro periodo de referéncia (por ejemplo, 1991-2020), de modo que disponga de un caudal medio mensual a largo plazo para enero, febrero... diciembre:

In [ ]:
# Converting date strings to datetime format
dt_start = datetime(clim_startYear, 1, 1)
dt_end = datetime(clim_endYear, 12, 31)

# Get data for the period of interest
monthly_clim = df_month[(df_month.index >= dt_start) &
                        (df_month.index <= dt_end)]

# Compute average for each calendar month
month_nums = monthly_clim.index.month
clim_mean = monthly_clim.groupby(month_nums).mean()

print('Promedio mensual:')
display(clim_mean)

4.2. Podemos comprobar si en cada mes del calendario hay suficientes años en el periodo de referencia para calcular los percentiles. Sería recomendable tener al menos 20 años de datos (30 es el ideal) para un cálculo adecuado:


In [ ]:
# Number of years used to compute monthly means
print(f'Periodo de referencia: {clim_startYear}-{clim_endYear}')
print(f'Número de años disponibles en cada mes p/ calculo de promedio mensual: {monthly_clim.groupby(monthly_clim.index.month).count()}')

4.3. Calculamos el caudal medio mensual como porcentaje de la media, utilizando el periodo de referéncia establecido:

In [ ]:
# Copy monthly flows and update it to % of average
df_prct_avg = df_month.copy()

# Compute % of average for each month separately
for i in range(1, 13):
  idx = (df_prct_avg.index.month == i)
  df_prct_avg.loc[idx] = 100 * (df_prct_avg.loc[idx] / clim_mean.loc[i].values)

4.4. Plotando el gráfico con los caudales representados como % del promedio mensual:

In [ ]:
#@title Grafico
# Plot on screen
df_prct_avg.plot(figsize=(12, 4),
                 linewidth=1.0,
                 title=f'% del promedio mensual, referéncia: {clim_startYear}-{clim_endYear}',
                 ylabel='Flujo como % del promedio',
                 legend=False)

---

# **5. Cálculo de percentiles y posición (rank)**


5.1. En esta etapa calcularemos los percentiles de posición (*rank percentiles*), utilizando los caudales calculados como % de la media en el paso anterior. En primer lugar, identificamos las posiciones (*ranks*) de la serie histórica que ocupan estos caudales. Al valor más bajo se le dará el *rank* 1, al segundo valor más bajo el *rank* 2, y así sucesivamente. Estos ranks son relativos al mes analizado, es decir, a los valores de enero se les asignarán *ranks* relativos a la serie histórica del mes de enero:

In [ ]:
# Compute number of months with data and ranks
df_n_months = df_prct_avg.groupby(df_prct_avg.index.month).count()
df_ranks = df_prct_avg.groupby(df_prct_avg.index.month).rank(na_option='keep')

# Keep ranks as float format to allow missing (NaN) values
print('Posición ordenada (rank) de los percentiles:')
display(df_ranks)

5.2. A continuación, calculamos los percentiles que se refieren a estos *ranks*. El método estándar del HydroSOS consiste en utilizar la distribución de Weibull:

\begin{align}
   \frac{i}{N+1}
\end{align}

Donde:

*   $ i $ es la posición (*rank*) del caudal;
*   $ N $ es el número de meses disponibles en el registro historico.

In [ ]:
# Creating a dataframe copy to update it afterwards
df_weibull_perc = df_ranks.copy()

# Compute Weibull rank percentiles for each month in the time series
for i in range(1, 13):
  idx = (df_prct_avg.index.month == i)
  df_weibull_perc.loc[idx] = df_ranks.loc[idx] / (df_n_months.loc[i].values + 1)

print('Percentiles de posición (rank percentiles) calculados:')
display(df_weibull_perc)

5.3. Aquí definiremos una función que retorna categorías numéricas (discretas) a partir de percentiles de posición (contínuos) informados. Estas categorías numéricas dependen de los percentiles definidos en la sección de parámetros de este tutorial:
*   `weibull_perc`: Percentil calculado con la distribución de Weibull
*   `cat_percentiles`: Lista con los percentiles superiores de cada categoría de flujo.

In [ ]:
# Defining the function to set flow categories
def flow_status(weibull_perc, cat_percentiles):
  status = np.nan
  # loop in the list of categories
  for i in range(len(cat_percentiles)):
    if weibull_perc <= cat_percentiles[i]:
      status = i + 1
      break

  return status

5.4. Podemos clasificar los percentiles en categorías numericas aplicando la función anterior a cada uno de los percentiles de posición calculados por la ecuación de Weibull:



In [ ]:
# Mapping the categorization function over the weibull rank percentiles
df_cat = df_weibull_perc.map(lambda x: flow_status(x, perc_categories))

print('Categoría numérica del caudal clasificado')
display(df_cat)

---

# **6. Generación del *status* HydroSOS**

6.1. Por último, creamos una tabla final con información relevante para el HydroSOS (fechas, caudales, % en relación con la media, orden (*rank*), percentiles y categorías):

In [ ]:
# Creating the final table concatenating data processed in previous steps
df_status = pd.concat([df_month, df_prct_avg, df_ranks, df_weibull_perc, df_cat], axis=1)
df_status.columns = ['flow','flow(%)','rank','rankperc','flowcat']

display(df_status)

6.2. Para facilitar la visualización de los resultados, podemos también mapear los nombres y colores de las categorías correspondientes a su clasificación numérica:

In [ ]:
# Linking category and colors to category numbers
link_names = dict(zip(np.arange(1,len(name_categories)+1), name_categories))
link_colors = dict(zip(np.arange(1,len(color_codes)+1), color_codes))

# Mapping category names using category numbers
df_status['cat_name'] = df_status['flowcat'].map(link_names)
df_status['cat_color'] = df_status['flowcat'].map(link_colors)

# For months with missing data, set white color to avoid errors
df_status.loc[df_status['cat_color'].isna(), 'cat_color'] = 'w'

display(df_status)

6.3. Finalmente visualizamos los resultados en formato categórico:

In [ ]:
#@title Grafico
# Plot the calculated status over the recent year
recent_months = 12

# Map the 'flowcat' column to the predefined colors
bar_colors = df_status['cat_color'].iloc[-recent_months:]

# Bar plot with flow percentiles
ax = df_status['rankperc'].iloc[-recent_months:].plot(figsize=(10,5),
                                                 kind='bar',
                                                 ylabel='Percentil relativo al mes',
                                                 xlabel='Fecha de estado',
                                                      edgecolor='k',
                                                 ylim=[0,1],
                                                 width=1,
                                                 color=bar_colors) # Set the color of the bars

# Create custom legend handles
legend_handles = []
for cat_num, cat_name in link_names.items():
    color = link_colors[cat_num]
    patch = mpatches.Patch(color=color, label=cat_name)
    legend_handles.append(patch)

# Add the legend above the plot
ax.legend(handles=legend_handles, bbox_to_anchor=(0.5, 1.15), loc='upper center', ncol=len(name_categories))

# Set date labels to YYYY-mm format to aid visualization
ax.set_xticklabels(df_status.index.strftime('%Y-%m').values[-recent_months:], rotation=45)

# Ensure the layout is tight to prevent labels overlapping
plt.tight_layout()

# **7. Exportando la tabla de estado**

In [ ]:
# Excluindo la columna de los colores
df_status_export = df_status.drop(columns=['cat_color'])

df_status_export.to_csv(f'{folder_output}/status.csv')

---

# **8. Calculando el estado usando el script del repositorio HydroSOS**

Esta sección muestra cómo calcular el estado hidrológico con la metodología HydroSOS utilizando el script `statuscalc.py` disponible en el repositorio de Github (https://github.com/wmo-im/HydroSOS). El script debe ser ejecutado con los siguientes parámetros:

`python statuscalc.py input_directory output_directory --startYear --endYear`

Donde:

* `input_directory`: es el directorio que contiene las series de caudal diário en formato .csv que se van a procesar. El script intentará analizar todos los archivos .csv en este directorio, por lo que se deben eliminar los archivos .csv que no se deseen procesar. Consulte los archivos en la carpeta `HydroSOS/example_data/status/input` para ver el **formato esperado** de estos archivos;

* `output_directory`: es el directorio donde se escribirán los archivos .csv procesados. Tendrán el mismo nombre que los archivos del directorio de entrada, pero con el prefijo "cat_" añadido al comienzo del nombre;

* `startYear`: es un argumento opcional que indica el año a utilizar como inicio del rango para calcular el caudal promedio de referencia;

* `endYear`: es un argumento opcional que indica el año a utilizar como final del rango para calcular el caudal promedio de referencia.

Para más información y scripts adicionales se recomienda consultar la documentación existente en en `HydroSOS/README.md`.

In [ ]:
# Create the folder for output files if it doesn't exist
os.makedirs('output_hydrosos', exist_ok=True)

# Now run the script using shell command
!python ./HydroSOS/status/statuscalc.py ./HydroSOS/example_data/status/input/ ./output_hydrosos/ --startYear 1991 --endYear 2020